# Inference - Klasifikasi Laporan Kerusakan Fasilitas Kampus
---
Notebook ini digunakan untuk **memuat model yang sudah dilatih** dan melakukan prediksi.

**Requirements:**
- Folder `models/` hasil training (dari notebook `train_klasifikasi_laporan.ipynb`)
- `transformers`, `torch`, `pickle`

In [ ]:
import pickle
import json
from pathlib import Path

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

In [ ]:
# ============================================================
# KONFIGURASI - Sesuaikan path dengan lokasi folder models/
# ============================================================
MODEL_DIR = Path("models")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
print(f"Model dir: {MODEL_DIR.resolve()}")

In [ ]:
# ============================================================
# LOAD ARTIFACTS
# ============================================================
# Load label encoders
with open(MODEL_DIR / "label_encoder_kategori.pkl", "rb") as f:
    le_kategori = pickle.load(f)
with open(MODEL_DIR / "label_encoder_urgensi.pkl", "rb") as f:
    le_urgensi = pickle.load(f)

print("Kategori classes:", le_kategori.classes_)
print("Urgensi classes:", le_urgensi.classes_)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR / "tokenizer"))
print(f"Tokenizer vocab size: {tokenizer.vocab_size}")

# Load models
model_kategori = AutoModelForSequenceClassification.from_pretrained(
    str(MODEL_DIR / "kategori")
).to(DEVICE).eval()

model_urgensi = AutoModelForSequenceClassification.from_pretrained(
    str(MODEL_DIR / "urgensi")
).to(DEVICE).eval()

print(f"\nModels loaded successfully!")
print(f"  Kategori model: {model_kategori.config.num_labels} classes")
print(f"  Urgensi model:  {model_urgensi.config.num_labels} classes")

# Load metadata (optional)
with open(MODEL_DIR / "metadata.json", "r") as f:
    metadata = json.load(f)
print(f"\n--- Model Performance ---")
print(f"Test F1 (Kategori): {metadata['test_f1_weighted_kategori']:.4f}")
print(f"Test F1 (Urgensi):  {metadata['test_f1_weighted_urgensi']:.4f}")

In [ ]:
# ============================================================
# PREDICTION FUNCTION
# ============================================================
def predict(text):
    """
    Predict kategori and urgency for a given report text.

    Parameters
    ----------
    text : str
        The report text in Indonesian.

    Returns
    -------
    dict
        {
            "kategori": str,
            "urgensi": str,
            "confidence_kategori": float,
            "confidence_urgensi": float,
            "probabilities_kategori": {class: prob, ...},
            "probabilities_urgensi": {class: prob, ...},
        }
    """
    if not isinstance(text, str) or not text.strip():
        return {"error": "Input must be a non-empty string"}

    clean = text.strip().lower()

    inputs = tokenizer(
        clean,
        truncation=True,
        padding="max_length",
        max_length=256,
        return_tensors="pt",
    ).to(DEVICE)

    with torch.no_grad():
        out_k = model_kategori(**inputs)
        out_u = model_urgensi(**inputs)

    prob_k = torch.softmax(out_k.logits, dim=1).cpu().numpy()[0]
    prob_u = torch.softmax(out_u.logits, dim=1).cpu().numpy()[0]

    idx_k = prob_k.argmax()
    idx_u = prob_u.argmax()

    return {
        "kategori": le_kategori.classes_[idx_k],
        "urgensi": le_urgensi.classes_[idx_u],
        "confidence_kategori": float(prob_k[idx_k]),
        "confidence_urgensi": float(prob_u[idx_u]),
        "probabilities_kategori": {
            str(cls): float(p) for cls, p in zip(le_kategori.classes_, prob_k)
        },
        "probabilities_urgensi": {
            str(cls): float(p) for cls, p in zip(le_urgensi.classes_, prob_u)
        },
    }


def predict_batch(texts):
    """Predict for a list of texts."""
    return [predict(t) for t in texts]


print("Prediction functions ready.")

In [ ]:
# ============================================================
# TEST SINGLE PREDICTION
# ============================================================
test_laporan = "Tolong mi, cermin besar di ruang ganti asrama pecah sudut kanannya tajam ngeri mengiris tangannya anak-anak pas lewat."

result = predict(test_laporan)
print("Input:", test_laporan[:100] + "...")
print("\nResult:")
print(json.dumps(result, indent=2, ensure_ascii=False))

In [ ]:
# ============================================================
# TEST BATCH PREDICTION
# ============================================================
test_samples = [
    "Bisa dicek kah router Wi-Fi di gazebo teknik? Lampunya merah terus dari pagi nda ada koneksi internet.",
    "Pak, kabel listrik yang melintang di atas area parkir itu terlalu turun, ngeri nyangkut di mobil tinggi.",
    "Tabe, handel pintu darurat lepas nda ada gantinya, macam mana kalau kita mau lari pas gempa?",
    "Pak, colokan LAN (wallplate) di kelas C3 masuk ke dalam dinding, nda bisa kita colok kabelnya.",
    "Sumpah lemot betul jaringan kampus buat akses jurnal internasional, time out terus dari tadi sa mau download PDF.",
    "Tolong diperbaiki kasian, figura foto presiden dan wakil presiden di ruang dosen kacanya pecah berantakan jatuh ke meja.",
]

print("=" * 60)
print("BATCH PREDICTION RESULTS")
print("=" * 60)

for i, text in enumerate(test_samples, 1):
    res = predict(text)
    print(f"\n[{i}] {text[:90]}...")
    print(f"    -> Kategori: {res['kategori']} (conf: {res['confidence_kategori']:.2%})")
    print(f"    -> Urgensi:  {res['urgensi']} (conf: {res['confidence_urgensi']:.2%})")

In [ ]:
# ============================================================
# EXPORT AS MODULE (FOR APPLICATION INTEGRATION)
# ============================================================
# You can save this class as a standalone Python file:
# klasifikasi_laporan.py

class KlasifikasiLaporan:
    """Main class for campus facility damage report classification.
    
    Usage:
        model = KlasifikasiLaporan("models")
        result = model.predict("AC ruang kelas mati, panas sekali")
        print(result["kategori"])  # -> "Kelistrikan & Tata Udara (MEP)"
        print(result["urgensi"])   # -> "Tinggi"
    """

    def __init__(self, model_dir="models"):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        model_path = Path(model_dir)

        with open(model_path / "label_encoder_kategori.pkl", "rb") as f:
            self.le_kategori = pickle.load(f)
        with open(model_path / "label_encoder_urgensi.pkl", "rb") as f:
            self.le_urgensi = pickle.load(f)

        self.tokenizer = AutoTokenizer.from_pretrained(str(model_path / "tokenizer"))

        self.model_kategori = AutoModelForSequenceClassification.from_pretrained(
            str(model_path / "kategori")
        ).to(self.device).eval()

        self.model_urgensi = AutoModelForSequenceClassification.from_pretrained(
            str(model_path / "urgensi")
        ).to(self.device).eval()

    def predict(self, text):
        if not isinstance(text, str) or not text.strip():
            return {"error": "Input must be a non-empty string"}

        clean = text.strip().lower()
        inputs = self.tokenizer(
            clean, truncation=True, padding="max_length",
            max_length=256, return_tensors="pt"
        ).to(self.device)

        with torch.no_grad():
            out_k = self.model_kategori(**inputs)
            out_u = self.model_urgensi(**inputs)

        prob_k = torch.softmax(out_k.logits, dim=1).cpu().numpy()[0]
        prob_u = torch.softmax(out_u.logits, dim=1).cpu().numpy()[0]

        idx_k = prob_k.argmax()
        idx_u = prob_u.argmax()

        return {
            "kategori": self.le_kategori.classes_[idx_k],
            "urgensi": self.le_urgensi.classes_[idx_u],
            "confidence_kategori": float(prob_k[idx_k]),
            "confidence_urgensi": float(prob_u[idx_u]),
            "probabilities_kategori": {
                str(cls): float(p)
                for cls, p in zip(self.le_kategori.classes_, prob_k)
            },
            "probabilities_urgensi": {
                str(cls): float(p)
                for cls, p in zip(self.le_urgensi.classes_, prob_u)
            },
        }

    def predict_batch(self, texts):
        return [self.predict(t) for t in texts]


# Test the class
clf = KlasifikasiLaporan("models")
res = clf.predict("AC ruang kelas mati total, panas sekali kasian kita belajar di dalam.")
print(json.dumps(res, indent=2, ensure_ascii=False))

---
### Integrasi ke Aplikasi

Simpan class `KlasifikasiLaporan` sebagai file `klasifikasi_laporan.py` dan gunakan:

```python
from klasifikasi_laporan import KlasifikasiLaporan

model = KlasifikasiLaporan("path/ke/folder/models")
result = model.predict("Laptop di lab rusak, layarnya mati")
print(result)
```

**Output:**
```json
{
  "kategori": "Teknologi Informasi & Jaringan (IT)",
  "urgensi": "Tinggi",
  "confidence_kategori": 0.98,
  "confidence_urgensi": 0.85,
  ...
}
```